In [ ]:
pip install --upgrade scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 53.6 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [ ]:
import sklearn
print(sklearn.__version__)


1.8.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
path="/content/iti_dropout_final.csv"

df=pd.read_csv(path)

In [ ]:
# Install optional gradient boosting libs
!pip -q install xgboost lightgbm catboost

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import joblib

In [ ]:
df.head()

,gender,age,program_enrolled,attendance_rate,backlogs,assignment_score,test_scores,dropout
0,Female,21,Ece,77.2,1,15.0,67.1,medium
1,Male,21,Electrical,60.8,0,15.0,47.5,medium
2,Male,20,Computer It,57.2,0,15.0,66.2,medium
3,Male,20,Plumbing,62.6,1,15.0,44.9,medium
4,Female,22,Ece,83.9,2,15.0,100.0,low


In [ ]:
features = ["attendance_rate", "test_scores", "backlogs", "assignment_score"]
target = "dropout"

data = df[features + [target]].copy()
print(data.shape)
print(data[target].value_counts())

(7000, 5)
dropout
medium    4503
high      1418
low       1079
Name: count, dtype: int64


In [ ]:
# Ensure numeric columns are numeric
for col in features:
    data[col] = pd.to_numeric(data[col], errors="coerce")

# Strip target text
data[target] = data[target].astype(str).str.strip().str.lower()

# Keep valid classes only
valid_classes = ["low", "medium", "high"]
data = data[data[target].isin(valid_classes)].copy()

print(data.isna().sum())

attendance_rate     0
test_scores         0
backlogs            0
assignment_score    0
dropout             0
dtype: int64


In [ ]:
le = LabelEncoder()
data["dropout_encoded"] = le.fit_transform(data[target])  # e.g., high/low/medium mapped to 0/1/2
print(dict(zip(le.classes_, le.transform(le.classes_))))

{'high': np.int64(0), 'low': np.int64(1), 'medium': np.int64(2)}


In [ ]:
X = data[features]
y = data["dropout_encoded"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
numeric_features = features

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[("num", numeric_transformer, numeric_features)]
)

In [ ]:
rf_model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        random_state=42,
        class_weight="balanced"
    ))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print("RF Accuracy:", accuracy_score(y_test, rf_pred))
print("RF Macro F1:", f1_score(y_test, rf_pred, average="macro"))
print(classification_report(y_test, rf_pred, target_names=le.classes_))

RF Accuracy: 0.9985714285714286
RF Macro F1: 0.9984484177530364
              precision    recall  f1-score   support

        high       1.00      0.99      1.00       283
         low       1.00      1.00      1.00       216
      medium       1.00      1.00      1.00       901

    accuracy                           1.00      1400
   macro avg       1.00      1.00      1.00      1400
weighted avg       1.00      1.00      1.00      1400



In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_pipe = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(random_state=42, class_weight="balanced"))
])

rf_param_dist = {
    "model__n_estimators": [200, 400, 600, 800],
    "model__max_depth": [None, 5, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", None]
}

rf_search = RandomizedSearchCV(
    rf_pipe,
    param_distributions=rf_param_dist,
    n_iter=30,
    scoring="f1_macro",
    cv=cv,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

rf_search.fit(X_train, y_train)
print("Best RF CV Macro F1:", rf_search.best_score_)
print("Best RF Params:", rf_search.best_params_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best RF CV Macro F1: 0.9984639646068947
Best RF Params: {'model__n_estimators': 400, 'model__min_samples_split': 2, 'model__min_samples_leaf': 1, 'model__max_features': None, 'model__max_depth': None}


In [ ]:
xgb_pipe = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", XGBClassifier(
        objective="multi:softmax",
        num_class=len(le.classes_),
        eval_metric="mlogloss",
        random_state=42
    ))
])

xgb_param_dist = {
    "model__n_estimators": [200, 400, 600],
    "model__max_depth": [3, 4, 5, 6, 8],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__subsample": [0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 0.9, 1.0]
}

xgb_search = RandomizedSearchCV(
    xgb_pipe,
    param_distributions=xgb_param_dist,
    n_iter=25,
    scoring="f1_macro",
    cv=cv,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

xgb_search.fit(X_train, y_train)
print("Best XGB CV Macro F1:", xgb_search.best_score_)
print("Best XGB Params:", xgb_search.best_params_)

Fitting 5 folds for each of 25 candidates, totalling 125 fits
Best XGB CV Macro F1: 0.9973155762973145
Best XGB Params: {'model__subsample': 0.8, 'model__n_estimators': 600, 'model__max_depth': 8, 'model__learning_rate': 0.1, 'model__colsample_bytree': 0.8}


In [ ]:
candidates = {
    "RF": rf_search.best_estimator_,
    "XGB": xgb_search.best_estimator_
}

best_name, best_model, best_f1 = None, None, -1

for name, model in candidates.items():
    pred = model.predict(X_test)
    f1 = f1_score(y_test, pred, average="macro")
    print(f"{name} Test Macro F1: {f1:.4f}")
    if f1 > best_f1:
        best_name, best_model, best_f1 = name, model, f1

print("Best model:", best_name, "with Macro F1:", best_f1)

final_pred = best_model.predict(X_test)
print(classification_report(y_test, final_pred, target_names=le.classes_))
print(confusion_matrix(y_test, final_pred))

RF Test Macro F1: 0.9992
XGB Test Macro F1: 0.9967
Best model: RF with Macro F1: 0.9992251524417678
              precision    recall  f1-score   support

        high       1.00      1.00      1.00       283
         low       1.00      1.00      1.00       216
      medium       1.00      1.00      1.00       901

    accuracy                           1.00      1400
   macro avg       1.00      1.00      1.00      1400
weighted avg       1.00      1.00      1.00      1400

[[282   0   1]
 [  0 216   0]
 [  0   0 901]]


In [ ]:
joblib.dump(best_model, "dropout_best_model.pkl")
joblib.dump(le, "dropout_label_encoder.pkl")
print("Saved!")

Saved!


In [ ]:
# Example new input
new_data = pd.DataFrame([{
    "attendance_rate": 75.0,
    "test_scores": 88.0,
    "backlogs": 0,
    "assignment_score": 15.0
}])

pred_num = best_model.predict(new_data)[0]
pred_label = le.inverse_transform([pred_num])[0]
print("Predicted dropout risk:", pred_label)

Predicted dropout risk: low


In [ ]:
import sklearn
print(sklearn.__version__)

1.8.0
